## 1. Установка необходимых библиотек

In [ ]:
!pip install --upgrade featuretools >> None
!pip install --upgrade catboost >> None

## 2. Импорт библиотек и настройка

In [ ]:
import pandas as pd
import numpy as np

import optuna

from sklearn.model_selection import train_test_split

import gc

from catboost import CatBoostRegressor, Pool

import warnings
warnings.filterwarnings("ignore")

## 3. Загрузка данных

In [ ]:
transaction_file = '/kaggle/input/dataset-generated/dataset_generated_with_cats.csv'
dataset_generated_with_cats = pd.read_csv(transaction_file)

In [ ]:
target_file = '/kaggle/input/alfa-challenge/train.pa'
df_target = pd.read_parquet(target_file)

In [ ]:
print("Данные успешно загружены.")

## 4. Подготовка данных

In [ ]:
target = df_target[['client_num', 'target']]
data_for_model = dataset_generated_with_cats.merge(target, on='client_num', how='inner')
print("Признаки и целевая переменная объединены для модели.")

cat_features = data_for_model.select_dtypes(include=['object', 'category']).columns.tolist()

X = data_for_model.drop(['client_num', 'target'], axis=1)
y = data_for_model['target']

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print("Данные разделены на обучающую и валидационную выборки.")

## 5. Расчет весов классов

In [ ]:
unique_classes = np.sort(y_train.unique())
class_counts = y_train.value_counts()
total_samples = len(y_train)
class_weights = {c: total_samples / (len(unique_classes) * class_counts[c]) for c in unique_classes}
print("Веса классов рассчитаны.")

def map_class_weights(y_labels, class_weights):
    return y_labels.map(class_weights).values

weights_train = map_class_weights(y_train, class_weights)
weights_valid = map_class_weights(y_valid, class_weights)
print("Веса для выборок рассчитаны.")

## 6. Освобождение памяти и создание Pool для CatBoost

In [ ]:
# Освобождение памяти
del data_for_model
del X
del y
gc.collect()

train_pool = Pool(
    data=X_train,
    label=y_train,
    weight=weights_train,
    cat_features=cat_features
)

valid_pool = Pool(
    data=X_valid,
    label=y_valid,
    weight=weights_valid,
    cat_features=cat_features
)

# Освобождение памяти
del X_train
del X_valid
del y_train
del y_valid
gc.collect()

## 7. Оптимизация гиперпараметров с использованием Optuna

In [ ]:
print("\nОптимизация гиперпараметров с помощью Optuna...")

def objective(trial):
    params = {
        'loss_function': 'MAE',
        'eval_metric': 'MAE',
        'logging_level': 'Silent',
        'random_seed': 42,
        'task_type': 'CPU',
        'iterations': 1_000,
        'learning_rate': trial.suggest_loguniform('learning_rate', 0.001, 0.1),
        'depth': trial.suggest_int('depth', 3, 12),
        'l2_leaf_reg': trial.suggest_int('l2_leaf_reg', 1, 50.0),
        "colsample_bylevel": trial.suggest_float("colsample_bylevel", 0.01, 0.8),
        'border_count': trial.suggest_int('border_count', 5, 255),
        'boosting_type': trial.suggest_categorical('boosting_type', ['Plain', 'Ordered']),
        'bootstrap_type': trial.suggest_categorical('bootstrap_type', ['Bayesian', 'Bernoulli', 'MVS', 'No']),
        'grow_policy': trial.suggest_categorical('grow_policy', ['SymmetricTree', 'Depthwise', 'Lossguide']),
    }

    if params['boosting_type'] == 'Ordered':
        params['grow_policy'] = 'SymmetricTree'
    else:
        params['grow_policy'] = trial.suggest_categorical('grow_policy', ['SymmetricTree', 'Depthwise', 'Lossguide'])
    
    if params['bootstrap_type'] == 'Bayesian':
        params['bagging_temperature'] = trial.suggest_float('bagging_temperature', 0.01, 20)
    elif params['bootstrap_type'] == 'Bernoulli':
        params['subsample'] = trial.suggest_float('subsample', 0.1, 1)

    model = CatBoostRegressor(
        **params
    )

    model.fit(
        train_pool,
        eval_set=valid_pool,
        verbose=False,
        early_stopping_rounds=50
    )

    preds = model.predict(valid_pool)
    wmae = np.sum(valid_pool.get_weight() * np.abs(valid_pool.get_label() - preds)) / np.sum(valid_pool.get_weight())
    return wmae

study = optuna.create_study(direction='minimize')

study.optimize(objective, timeout=10*60*60) # максимум - 10 часов

print("Наилучшие гиперпараметры:")
print(study.best_params)

## 8. Обучение модели с лучшими гиперпараметрами

In [ ]:
best_params = study.best_params
best_params.update({
    'loss_function': 'MAE',
    'eval_metric': 'MAE',
    'random_seed': 42,
    'verbose': 50
})

best_model = CatBoostRegressor(
    **best_params
)

best_model.fit(
    train_pool,
    eval_set=valid_pool,
    early_stopping_rounds=100
)

y_valid_pred = best_model.predict(valid_pool)
wmae = np.sum(valid_pool.get_weight() * np.abs(valid_pool.get_label() - y_valid_pred)) / np.sum(valid_pool.get_weight())
print('CatBoost Validation WMAE:', wmae)

## 9. Результаты обучения

In [ ]:
model_results = {
    'Model': ['CatBoost'],
    'Hyperparameters': [best_params],
    'Score (WMAE)': [wmae]
}

results_df = pd.DataFrame(model_results)

print("\nРезультаты:")
print(results_df)